In [1]:
# Import Libraries
import numpy as np 
from datetime import datetime
from tabulate import tabulate
from Black_Scholes import BS
import pandas as pd

## Newton Method

In [2]:
def newton_iv(className, spot, strike, rate, dte, callprice=None, putprice=None):
    x0 = 1               # initial guess
    h = 0.001            # step size    
    tolerance = 1e-7     # 7-digit accuracy is desired
    epsilon = 1e-14      #  do not divide by a number smaller than this, some kind of error / floor
    maxiter = 200        # maximum number of iterations to execute

    # function whose root we are trying to find
    # f(x) = Black Scholes Call price - Market Price - defining the f(x) here 
    if callprice:
        f = lambda x: eval(className)(spot, strike, rate, dte, x).callPrice - callprice
    if putprice:
       f = lambda x: eval(className)(spot, strike, rate, dte, x).putPrice - putprice
    for i in range(maxiter):
        y = f(x0)                           # starting with initial guess
        yprime = (f(x0+h) - f(x0-h))/(2*h)  # central difference, the derivative of the function
        if abs(yprime)<epsilon: # stop if the denominator is too small
           break
        x1 = x0 - y/yprime      # perform Newton's computation
        
        if (abs(x1-x0) <= tolerance*abs(x1)): # stop when the result is within the desired tolerance
            break
        x0=x1                                   # update x0 to start the process
        
    return x1 # x1 is a solution within tolerance and maximum number of iterations

In [3]:
# newton iv
newton_iv('BS',100,100,0.02,1,callprice=8)

0.17657213831399154

In [4]:
#Calculating the Implied Vol of options presented in the exercise by Paul Wilmott in his Understanding of Volatility lecture. 
Expiry = [0.25, 0.5, 0.75, 1]
Value = [4.74, 6.72, 8.22, 9.63]
Implied_Vol = []
for i in range (len(Expiry)):
    Implied_Vol.append(newton_iv('BS', 108.5, 110,0.04,Expiry[i],callprice=Value[i]))

dfa = pd.DataFrame({'Tenor':Expiry, 'Value': Value, 'Implied Volatility':Implied_Vol})
header3a = ['Tenors', 'Value', 'Implied Volatility']
print(tabulate(dfa, tablefmt="fancy_outline", headers=header3a, numalign='center', floatfmt=("#", ".3f", ".3f", ".3%")))

╒════╤══════════╤═════════╤══════════════════════╕
│    │  Tenors  │  Value  │  Implied Volatility  │
╞════╪══════════╪═════════╪══════════════════════╡
│ 0  │  0.250   │  4.740  │       22.796%        │
│ 1  │  0.500   │  6.720  │       20.913%        │
│ 2  │  0.750   │  8.220  │       19.687%        │
│ 3  │  1.000   │  9.630  │       19.097%        │
╘════╧══════════╧═════════╧══════════════════════╛


In [20]:
#Calculating the Implied Vol of options presented in the exercise by Paul Wilmott in his Understanding of Volatility lecture. 
Expiry2 = [1/12, 3/12, 7/12]
Value2 = [11.80, 13.33, 16.02]
Implied_Vol2 = []
for i in range (len(Expiry2)):
    Implied_Vol2.append(newton_iv('BS', 100, 105,0.04,Expiry[i],callprice=Value[i]))

dfb = pd.DataFrame({'Tenor':Expiry2, 'Value': Value2, 'Implied Volatility':Implied_Vol2})
header3b = ['Tenors', 'Value', 'Implied Volatility']
print(tabulate(dfb, tablefmt="fancy_outline", headers=header3b, numalign='center', floatfmt=("#", ".3f", ".3f", ".3%")))

╒════╤══════════╤═════════╤══════════════════════╕
│    │  Tenors  │  Value  │  Implied Volatility  │
╞════╪══════════╪═════════╪══════════════════════╡
│ 0  │  0.083   │ 11.800  │       32.130%        │
│ 1  │  0.250   │ 13.330  │       28.338%        │
│ 2  │  0.583   │ 16.020  │       26.255%        │
╘════╧══════════╧═════════╧══════════════════════╛


## Bisection Method

In [5]:
# Bisection Method
def bisection_iv(className, spot, strike, rate, dte, callprice=None,putprice=None, high=500.0, low=0.0): 

    # this is market price
    if callprice:
        price = callprice
    if putprice and not callprice:
        price = putprice
    tolerance = 1e-7
    for i in range(1000):
        mid = (high + low) / 2              # c= (a+b)/2
        if mid < tolerance:
            mid = tolerance
        if callprice:
            estimate = eval(className)(spot, strike, rate, dte, mid).callPrice #␣Blackscholes price
        if putprice:
            estimate = eval(className)(spot, strike, rate, dte, mid).putPrice
        if round(estimate,6) == price:
            break
        elif estimate > price:
            high = mid                      # replace c with b | b = c
        elif estimate < price:
            low = mid                       # replace c with a | a = c
    return mid

In [6]:
# bisection iv
bisection_iv('BS',100,100,0.02,1,callprice=8)

0.17657213902566582

## BS Implied Volatility

In [7]:
# Initialize option
option = BS(100,100,0.05,1,0.2, callprice=8)
header = ['Option Price', 'Delta', 'Gamma', 'Theta', 'Vega', 'Rho', 'ImpVol']
table = [[option.callPrice, option.callDelta, option.gamma, option.callTheta,option.vega, option.callRho, option.impvol]]
print(tabulate(table,header))

  Option Price     Delta     Gamma       Theta     Vega       Rho    ImpVol
--------------  --------  --------  ----------  -------  --------  --------
       10.4506  0.636831  0.018762  -0.0175727  0.37524  0.532325  0.133776
